# Phase 04O: Teacher C → S3 Direct Logit-KD GPU Preflight

Notebook xác minh implementation Direct Logit-KD giữa Teacher C SigLIP
ViT-B/16 và Student S3 TinyViT-5M trước production training.

- Teacher C dùng checkpoint tốt nhất seed 42, luôn `eval/frozen/no_grad`.
- Student S3 khởi tạo lại từ shared Phase 02A seed-42 checkpoints.
- Loss khóa bởi Phase 04N: `(1-alpha) CE + alpha T² KL`, `T=4`, `alpha=0.5`.
- Chỉ Train+Restval được đọc; Held-out Test tiếp tục `SEALED`.
- Các optimizer step chỉ phục vụ preflight và bị loại bỏ.


## Cell 1 — Environment, authorization và leakage boundary

Xác minh GPU, deterministic mode, Phase 04N completion/authorization,
Teacher C seed 42, Teacher Pool, Phase 02A và Phase 01. Chưa đọc dữ liệu,
checkpoint payload hoặc Held-out Test.


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import time
from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase4_teacher_c_s3_logit_kd_gpu_preflight"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
RESULT_DIR = OUTPUT_ROOT / "results"
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    AUDIT_DIR,
    CONFIG_DIR,
    RESULT_DIR,
    CHECKPOINT_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


assert torch.cuda.is_available(), "Phase 04O yêu cầu GPU."
set_deterministic(SEED)
device = torch.device("cuda")

PHASE1_READY = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
PHASE2_READY = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
TEACHER_C_RUN_COMPLETE = exactly_one(
    "TEACHER_C_SEED42_FULL_RUN_COMPLETE.json"
)
TEACHER_POOL_COMPLETE = exactly_one(
    "PHASE3_TEACHER_POOL_COMPLETE.json"
)
PHASE4N_COMPLETE = exactly_one(
    "PHASE4_TEACHER_C_S3_KD_COMPATIBILITY_ENTRY_COMPLETE.json"
)
PHASE4O_AUTHORIZED = exactly_one(
    "PHASE4_TEACHER_C_S3_LOGIT_KD_PREFLIGHT_AUTHORIZED.json"
)

phase2_ready = read_json(PHASE2_READY)
teacher_run_marker = read_json(TEACHER_C_RUN_COMPLETE)
teacher_pool_marker = read_json(TEACHER_POOL_COMPLETE)
phase4n_complete = read_json(PHASE4N_COMPLETE)
phase4o_authorized = read_json(PHASE4O_AUTHORIZED)

assert phase2_ready["status"] == "ready"
assert phase2_ready["full_training_authorized"] is True
assert teacher_run_marker["status"] == "complete"
assert teacher_run_marker["teacher_id"] == "C"
assert int(teacher_run_marker["seed"]) == SEED
assert teacher_run_marker["quality_gate_passed"] is True
assert teacher_run_marker["qualified_for_kd"] is True
assert teacher_pool_marker["status"] == "complete"
assert teacher_pool_marker["all_teacher_runs_qualified_for_kd"] is True
assert phase4n_complete["status"] == "complete"
assert phase4n_complete["selected_method"] == "direct_logit_kd"
assert phase4n_complete["phase04o_authorized"] is True
assert phase4o_authorized["status"] == "authorized"
assert phase4o_authorized["experiment_id"] == "P4C_C_S3_LOGIT"
assert phase4o_authorized["teacher_id"] == "C"
assert phase4o_authorized["student_id"] == "S3"
assert phase4o_authorized["selected_method"] == "direct_logit_kd"
assert float(phase4o_authorized["temperature"]) == 4.0
assert float(phase4o_authorized["alpha"]) == 0.5

for marker in [
    teacher_run_marker,
    teacher_pool_marker,
    phase4n_complete,
    phase4o_authorized,
]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

write_json(
    AUDIT_DIR / "phase4o_cell01_environment_authorization.json",
    {
        "audit_version": "phase4o_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "seed": SEED,
        "strict_deterministic": True,
        "experiment_id": "P4C_C_S3_LOGIT",
        "phase4n_complete": str(PHASE4N_COMPLETE),
        "phase4o_authorized": str(PHASE4O_AUTHORIZED),
        "dataset_read": False,
        "caption_read": False,
        "checkpoint_payload_read": False,
        "optimizer_step_performed": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04O CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Experiment                : Teacher C -> S3 direct Logit-KD")
print("Development seed          : 42")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Strict deterministic      : True")
print("Phase 04N                 : VERIFIED")
print("Phase 04O authorization   : VERIFIED")
print("Checkpoint payload read   : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : verify inputs and checkpoints")


PHASE 04O CELL 1 STATUS
Status                    : PASSED
Experiment                : Teacher C -> S3 direct Logit-KD
Development seed          : 42
GPU                       : Tesla T4
Strict deterministic      : True
Phase 04N                 : VERIFIED
Phase 04O authorization   : VERIFIED
Checkpoint payload read   : False
Full training started     : False
Test Set status           : SEALED
Next                      : verify inputs and checkpoints


## Cell 2 — Manifest, checkpoint và offline SigLIP integrity

Hash lại Phase 04N và Phase 02A, khóa policy Direct Logit-KD, xác minh
Teacher C best checkpoint, S3 initialization checkpoints và package
`google/siglip-base-patch16-224` ngoại tuyến. Chưa deserialize checkpoint.


In [2]:
CELL_STARTED_AT = time.perf_counter()


def verify_manifest(root, csv_path, sums_path):
    root = Path(root).resolve()
    with Path(csv_path).open(
        "r", encoding="utf-8", newline=""
    ) as handle:
        rows = list(csv.DictReader(handle))
    expected_lines = []
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"], artifact
        expected_lines.append(
            f'{row["sha256"]}  {row["relative_path"]}'
        )
    actual_lines = [
        line.strip()
        for line in Path(sums_path).read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]
    assert actual_lines == expected_lines
    return rows


PHASE4N_ROOT = PHASE4N_COMPLETE.parent
phase4n_manifest_csv = resolve_inside(
    PHASE4N_ROOT,
    phase4n_complete["manifest"]["csv"]["relative_path"],
)
phase4n_sums = resolve_inside(
    PHASE4N_ROOT,
    phase4n_complete["manifest"]["sha256sums"]["relative_path"],
)
assert sha256_file(phase4n_manifest_csv) == (
    phase4n_complete["manifest"]["csv"]["sha256"]
)
assert sha256_file(phase4n_sums) == (
    phase4n_complete["manifest"]["sha256sums"]["sha256"]
)
phase4n_rows = verify_manifest(
    PHASE4N_ROOT, phase4n_manifest_csv, phase4n_sums
)
assert len(phase4n_rows) == 7

policy_ref = phase4o_authorized["policy"]
POLICY_PATH = resolve_inside(
    PHASE4N_ROOT, policy_ref["relative_path"]
)
assert sha256_file(POLICY_PATH) == policy_ref["sha256"]
LOGIT_KD_POLICY = read_json(POLICY_PATH)
COMPATIBILITY_PATH = resolve_inside(
    PHASE4N_ROOT,
    phase4n_complete["compatibility_decision"]["relative_path"],
)
assert sha256_file(COMPATIBILITY_PATH) == (
    phase4n_complete["compatibility_decision"]["sha256"]
)
COMPATIBILITY_CONTRACT = read_json(COMPATIBILITY_PATH)
assert LOGIT_KD_POLICY["status"] == "locked"
assert COMPATIBILITY_CONTRACT["status"] == "locked"
assert LOGIT_KD_POLICY["supervision"]["temperature"] == 4.0
assert LOGIT_KD_POLICY["supervision"]["alpha"] == 0.5
assert LOGIT_KD_POLICY["teacher"]["mode"] == "eval_frozen_no_grad"
LOSS_CONTRACT = {
    "loss": {
        "temperature": float(
            LOGIT_KD_POLICY["supervision"]["temperature"]
        ),
        "alpha": float(
            LOGIT_KD_POLICY["supervision"]["alpha"]
        ),
    },
    "compatibility": {
        "pad_id": int(
            COMPATIBILITY_CONTRACT[
                "teacher"
            ]["special_token_ids"]["pad"]
        ),
    },
}

PHASE2_ROOT = PHASE2_READY.parent
phase2_manifest_csv = resolve_inside(
    PHASE2_ROOT, phase2_ready["manifests"]["csv"]["relative_path"]
)
phase2_sums = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["manifests"]["sha256sums"]["relative_path"],
)
assert sha256_file(phase2_manifest_csv) == (
    phase2_ready["manifests"]["csv"]["sha256"]
)
assert sha256_file(phase2_sums) == (
    phase2_ready["manifests"]["sha256sums"]["sha256"]
)
phase2_rows = verify_manifest(
    PHASE2_ROOT, phase2_manifest_csv, phase2_sums
)
assert len(phase2_rows) == 38

S3_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s3_encoder_checkpoint"],
)
SHARED_DECODER_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["shared_decoder_checkpoint"],
)
SHARED_DECODER_CONTRACT = exactly_one(
    "shared_transformer_decoder_contract.json"
)
S3_ENCODER_CONTRACT = exactly_one(
    "s3_encoder_adapter_contract.json"
)

teacher_pool_root = TEACHER_POOL_COMPLETE.parent
teacher_registry_path = resolve_inside(
    teacher_pool_root,
    teacher_pool_marker[
        "teacher_c_checkpoint_registry"
    ]["relative_path"],
)
assert sha256_file(teacher_registry_path) == (
    teacher_pool_marker[
        "teacher_c_checkpoint_registry"
    ]["sha256"]
)
teacher_registry = read_json(teacher_registry_path)
seed42_entries = [
    item for item in teacher_registry["entries"]
    if int(item["seed"]) == SEED
]
assert len(seed42_entries) == 1
teacher_seed42_registry = seed42_entries[0]
TEACHER_C_BEST_CHECKPOINT = exactly_one(
    "teacher_c_seed42_best.pt"
)
assert sha256_file(TEACHER_C_BEST_CHECKPOINT) == (
    teacher_seed42_registry["best_checkpoint"]["sha256"]
)
assert sha256_file(TEACHER_C_BEST_CHECKPOINT) == (
    LOGIT_KD_POLICY["teacher"]["checkpoint_sha256"]
)

candidate_manifests = []
for manifest_path in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
    try:
        manifest = read_json(manifest_path)
    except Exception:
        continue
    if manifest.get("repo_id") == "google/siglip-base-patch16-224":
        candidate_manifests.append((manifest_path.resolve(), manifest))
assert len(candidate_manifests) == 1, candidate_manifests
SIGLIP_SOURCE_MANIFEST_PATH, SIGLIP_SOURCE_MANIFEST = (
    candidate_manifests[0]
)
SIGLIP_SOURCE_DIR = SIGLIP_SOURCE_MANIFEST_PATH.parent
assert SIGLIP_SOURCE_MANIFEST["schema_version"] == (
    "offline_huggingface_model_source_v1"
)
assert SIGLIP_SOURCE_MANIFEST["resolved_commit"] == (
    LOGIT_KD_POLICY["teacher"]["revision"]
)
source_rows = SIGLIP_SOURCE_MANIFEST["files"]
assert source_rows
for row in source_rows:
    artifact = resolve_inside(
        SIGLIP_SOURCE_DIR, row["relative_path"]
    )
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]
assert (SIGLIP_SOURCE_DIR / "config.json").is_file()
assert (SIGLIP_SOURCE_DIR / "preprocessor_config.json").is_file()
assert list(SIGLIP_SOURCE_DIR.glob("*.safetensors")) or list(
    SIGLIP_SOURCE_DIR.glob("pytorch_model*.bin")
)

phase2_by_path = {
    row["relative_path"]: row for row in phase2_rows
}
for checkpoint in [
    S3_INITIAL_CHECKPOINT,
    SHARED_DECODER_INITIAL_CHECKPOINT,
]:
    relative = checkpoint.relative_to(PHASE2_ROOT).as_posix()
    assert relative in phase2_by_path
    assert sha256_file(checkpoint) == phase2_by_path[relative]["sha256"]

write_json(
    AUDIT_DIR / "phase4o_cell02_input_integrity.json",
    {
        "audit_version": "phase4o_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase4n_manifest_rows": len(phase4n_rows),
        "phase2_manifest_rows": len(phase2_rows),
        "teacher_checkpoint": str(TEACHER_C_BEST_CHECKPOINT),
        "teacher_checkpoint_sha256": sha256_file(
            TEACHER_C_BEST_CHECKPOINT
        ),
        "s3_checkpoint_sha256": sha256_file(
            S3_INITIAL_CHECKPOINT
        ),
        "decoder_checkpoint_sha256": sha256_file(
            SHARED_DECODER_INITIAL_CHECKPOINT
        ),
        "siglip_source": str(SIGLIP_SOURCE_DIR),
        "siglip_source_manifest_rows": len(source_rows),
        "checkpoint_payload_read": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04O CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 04N manifest        : 7 / 7 VERIFIED")
print("Phase 02A manifest        : 38 / 38 VERIFIED")
print("Teacher C registry        : VERIFIED")
print("Teacher C best checkpoint : VERIFIED")
print("S3 initial checkpoint     : VERIFIED")
print("Shared decoder checkpoint : VERIFIED")
print("Offline SigLIP source     : VERIFIED")
print("Source manifest rows      :", len(source_rows))
print("Checkpoint payload read   : False")
print("Test Set status           : SEALED")
print("Next                      : build fixed Train preflight batch")


PHASE 04O CELL 2 STATUS
Status                    : PASSED
Phase 04N manifest        : 7 / 7 VERIFIED
Phase 02A manifest        : 38 / 38 VERIFIED
Teacher C registry        : VERIFIED
Teacher C best checkpoint : VERIFIED
S3 initial checkpoint     : VERIFIED
Shared decoder checkpoint : VERIFIED
Offline SigLIP source     : VERIFIED
Source manifest rows      : 4
Checkpoint payload read   : False
Test Set status           : SEALED
Next                      : build fixed Train preflight batch


## Cell 3 — Fixed Train batch và dual preprocessing

Tạo batch cố định gồm tám ảnh Train+Restval, không augmentation. Teacher
dùng đúng eval geometry SigLIP resize-224/center-crop-224 và normalization
`(0.5,0.5,0.5)`; Student dùng resize-256/center-crop-224 cùng ImageNet
normalization. Chỉ đọc `token_ids`, không đọc caption text hoặc Test.


In [3]:
CELL_STARTED_AT = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode


PHASE1_ROOT = PHASE1_READY.parent
TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")


def find_image_leaf(directory_name, prefix):
    candidates = []
    for path in INPUT_ROOT.rglob(directory_name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            candidates.append(path.resolve())
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, candidates
    return candidates[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

manifest_table = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id",
        "experiment_split",
        "source_partition",
        "image_path",
    ],
    filters=[("experiment_split", "=", "train")],
)
train_paths = {}
for row in manifest_table.to_pylist():
    image_id = int(row["image_id"])
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    path = leaf / Path(row["image_path"]).name
    train_paths[image_id] = path
assert len(train_paths) == 113287

fixed_records = []
seen_images = set()
caption_file = pq.ParquetFile(TRAIN_PARQUET)
for record_batch in caption_file.iter_batches(
    batch_size=1024,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in record_batch.to_pylist():
        image_id = int(row["image_id"])
        if image_id in seen_images:
            continue
        token_ids = [int(value) for value in row["token_ids"]]
        assert token_ids[0] == 1 and token_ids[-1] == 2
        fixed_records.append(
            {
                "annotation_id": int(row["annotation_id"]),
                "image_id": image_id,
                "image_path": train_paths[image_id],
                "token_ids": token_ids,
            }
        )
        seen_images.add(image_id)
        if len(fixed_records) == 8:
            break
    if len(fixed_records) == 8:
        break
assert len(fixed_records) == 8
assert len({row["image_id"] for row in fixed_records}) == 8
assert all(row["image_path"].is_file() for row in fixed_records)

teacher_transform = transforms.Compose(
    [
        transforms.Resize(
            224, interpolation=InterpolationMode.BICUBIC
        ),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
        ),
    ]
)
student_transform = transforms.Compose(
    [
        transforms.Resize(
            256, interpolation=InterpolationMode.BICUBIC
        ),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]
)


class FixedKDPreflightDataset(Dataset):
    def __len__(self):
        return len(fixed_records)

    def __getitem__(self, index):
        record = fixed_records[index]
        with Image.open(record["image_path"]) as image:
            rgb = image.convert("RGB")
            teacher_image = teacher_transform(rgb)
            student_image = student_transform(rgb)
        ids = torch.tensor(record["token_ids"], dtype=torch.long)
        return {
            "image_id": record["image_id"],
            "annotation_id": record["annotation_id"],
            "teacher_image": teacher_image,
            "student_image": student_image,
            "token_ids": ids,
        }


def kd_collate(rows):
    maximum = max(item["token_ids"].numel() for item in rows)
    full = torch.zeros(len(rows), maximum, dtype=torch.long)
    for index, item in enumerate(rows):
        length = item["token_ids"].numel()
        full[index, :length] = item["token_ids"]
    decoder_input_ids = full[:, :-1]
    labels = full[:, 1:]
    return {
        "image_ids": torch.tensor(
            [item["image_id"] for item in rows], dtype=torch.long
        ),
        "annotation_ids": torch.tensor(
            [item["annotation_id"] for item in rows], dtype=torch.long
        ),
        "teacher_images": torch.stack(
            [item["teacher_image"] for item in rows]
        ),
        "student_images": torch.stack(
            [item["student_image"] for item in rows]
        ),
        "decoder_input_ids": decoder_input_ids,
        "decoder_attention_mask": decoder_input_ids.ne(0),
        "labels": labels,
    }


fixed_loader = DataLoader(
    FixedKDPreflightDataset(),
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=kd_collate,
)
FIXED_BATCH = next(iter(fixed_loader))
assert FIXED_BATCH["teacher_images"].shape == (8, 3, 224, 224)
assert FIXED_BATCH["student_images"].shape == (8, 3, 224, 224)
assert FIXED_BATCH["decoder_input_ids"].shape == FIXED_BATCH["labels"].shape
assert int(FIXED_BATCH["labels"].ne(0).sum()) > 0

write_json(
    AUDIT_DIR / "phase4o_cell03_fixed_train_batch.json",
    {
        "audit_version": "phase4o_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "image_count": 8,
        "unique_images": 8,
        "teacher_tensor_shape": list(
            FIXED_BATCH["teacher_images"].shape
        ),
        "student_tensor_shape": list(
            FIXED_BATCH["student_images"].shape
        ),
        "decoder_shape": list(
            FIXED_BATCH["decoder_input_ids"].shape
        ),
        "augmentation": "disabled",
        "caption_text_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04O CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Fixed Train images        : 8 / 8")
print("Unique images             : 8")
print("Teacher tensor            :", list(FIXED_BATCH["teacher_images"].shape))
print("Student tensor            :", list(FIXED_BATCH["student_images"].shape))
print("Decoder tensor            :", list(FIXED_BATCH["decoder_input_ids"].shape))
print("Augmentation              : DISABLED")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Next                      : reconstruct and strict-load models")


PHASE 04O CELL 3 STATUS
Status                    : PASSED
Fixed Train images        : 8 / 8
Unique images             : 8
Teacher tensor            : [8, 3, 224, 224]
Student tensor            : [8, 3, 224, 224]
Decoder tensor            : [8, 13]
Augmentation              : DISABLED
Caption text read         : False
Test Parquet opened       : False
Test images read          : False
Test Set status           : SEALED
Next                      : reconstruct and strict-load models


## Cell 4 — Reconstruct Teacher C và Student S3

Dựng SigLIP ViT-B/16 + adapter 196-token cho Teacher và TinyViT-5M +
adapter 49-token cho Student. Strict-load năm thành phần; Teacher bị đóng
băng. Hai nhánh có visual memory khác chiều dài nhưng logits phải cùng
`[B,T,10080]`.


In [4]:
CELL_STARTED_AT = time.perf_counter()
import timm
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from transformers import SiglipModel


decoder_config = read_json(
    SHARED_DECODER_CONTRACT
)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080
assert int(decoder_config["pad_id"]) == 0
assert int(decoder_config["maximum_decoder_steps"]) == 31


class SharedCaptionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.vocabulary_size = int(config["vocabulary_size"])
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(
            config["maximum_decoder_steps"]
        )
        self.token_embedding = nn.Embedding(
            self.vocabulary_size,
            self.d_model,
            padding_idx=self.pad_id,
        )
        self.position_embedding = nn.Embedding(
            self.maximum_decoder_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(
            float(config["dropout"])
        )
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation=config["activation"],
            batch_first=True,
            norm_first=True,
            bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(
            torch.zeros(self.vocabulary_size)
        )

    def forward(
        self,
        decoder_input_ids,
        visual_memory,
        decoder_attention_mask=None,
    ):
        batch, length = decoder_input_ids.shape
        positions = torch.arange(
            length, device=decoder_input_ids.device
        ).unsqueeze(0).expand(batch, -1)
        target = (
            self.token_embedding(decoder_input_ids)
            * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=decoder_input_ids.device,
            ),
            diagonal=1,
        )
        padding = (
            None
            if decoder_attention_mask is None
            else ~decoder_attention_mask.bool()
        )
        hidden = self.transformer_decoder(
            tgt=target,
            memory=visual_memory,
            tgt_mask=causal,
            tgt_key_padding_mask=padding,
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden,
            self.token_embedding.weight,
            self.output_bias,
        )


class TeacherVisualAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Linear(768, 256)
        self.position_embedding = nn.Parameter(
            torch.zeros(1, 196, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, patches):
        assert patches.shape[1:] == (196, 768)
        return self.output_norm(
            self.projection(patches) + self.position_embedding
        )


def select_tensor_from_feature_output(feature_output):
    if torch.is_tensor(feature_output):
        return feature_output
    if isinstance(feature_output, dict):
        tensors = [
            value for value in feature_output.values()
            if torch.is_tensor(value)
        ]
        assert tensors
        return tensors[-1]
    if isinstance(feature_output, (tuple, list)):
        tensors = [
            value for value in feature_output
            if torch.is_tensor(value)
        ]
        assert tensors
        return tensors[-1]
    raise TypeError(type(feature_output).__name__)


class StudentVisualEncoderAdapter(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        self.projection = nn.Linear(320, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = select_tensor_from_feature_output(
            self.backbone.forward_features(images)
        )
        assert tuple(feature.shape[1:]) == (320, 7, 7)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 320)
        return self.output_norm(
            self.projection(tokens)
            + self.visual_position_embedding
        )


os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
full_siglip = SiglipModel.from_pretrained(
    SIGLIP_SOURCE_DIR, local_files_only=True
)
TEACHER_ENCODER = full_siglip.vision_model
del full_siglip
TEACHER_ADAPTER = TeacherVisualAdapter()
TEACHER_DECODER = SharedCaptionDecoder(decoder_config)

tinyvit_backbone = timm.create_model(
    "tiny_vit_5m_224.dist_in22k_ft_in1k",
    pretrained=False,
)
if hasattr(tinyvit_backbone, "reset_classifier"):
    tinyvit_backbone.reset_classifier(
        num_classes=0, global_pool=""
    )
removed_heads = []
for head_name in ["head", "head_dist", "classifier", "fc"]:
    module = getattr(tinyvit_backbone, head_name, None)
    if (
        isinstance(module, nn.Module)
        and not isinstance(module, nn.Identity)
    ):
        setattr(tinyvit_backbone, head_name, nn.Identity())
        removed_heads.append(head_name)
assert removed_heads == ["head"]
STUDENT_ENCODER = StudentVisualEncoderAdapter(tinyvit_backbone)
STUDENT_DECODER = SharedCaptionDecoder(decoder_config)

with torch.serialization.safe_globals([TorchVersion]):
    teacher_checkpoint = torch.load(
        TEACHER_C_BEST_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
    student_encoder_checkpoint = torch.load(
        S3_INITIAL_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
    student_decoder_checkpoint = torch.load(
        SHARED_DECODER_INITIAL_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )

assert teacher_checkpoint["schema_version"] == (
    "phase3_teacher_c_seed42_checkpoint_v1"
)
assert teacher_checkpoint["teacher_id"] == "C"
assert int(teacher_checkpoint["seed"]) == SEED
assert teacher_checkpoint["test_set_status"] == "sealed"
assert teacher_checkpoint["test_set_used"] is False

teacher_encoder_state = TEACHER_ENCODER.state_dict()
unknown_teacher_keys = set(
    teacher_checkpoint["encoder_candidate_state"]
) - set(teacher_encoder_state)
assert not unknown_teacher_keys, unknown_teacher_keys
teacher_encoder_state.update(
    teacher_checkpoint["encoder_candidate_state"]
)
TEACHER_ENCODER.load_state_dict(
    teacher_encoder_state, strict=True
)
teacher_adapter_load = TEACHER_ADAPTER.load_state_dict(
    teacher_checkpoint["adapter_state"], strict=True
)
teacher_decoder_load = TEACHER_DECODER.load_state_dict(
    teacher_checkpoint["decoder_state"], strict=True
)
student_encoder_load = STUDENT_ENCODER.load_state_dict(
    student_encoder_checkpoint["state_dict"], strict=True
)
student_decoder_load = STUDENT_DECODER.load_state_dict(
    student_decoder_checkpoint["state_dict"], strict=True
)
for result in [
    teacher_adapter_load,
    teacher_decoder_load,
    student_encoder_load,
    student_decoder_load,
]:
    assert not result.missing_keys
    assert not result.unexpected_keys

STUDENT_ENCODER_INITIAL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
STUDENT_DECODER_INITIAL_STATE = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}

del teacher_checkpoint
del student_encoder_checkpoint
del student_decoder_checkpoint
del teacher_encoder_state

TEACHER_ENCODER = TEACHER_ENCODER.to(device)
TEACHER_ADAPTER = TEACHER_ADAPTER.to(device)
TEACHER_DECODER = TEACHER_DECODER.to(device)
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

for module in [
    TEACHER_ENCODER,
    TEACHER_ADAPTER,
    TEACHER_DECODER,
]:
    module.eval()
    for parameter in module.parameters():
        parameter.requires_grad_(False)


def teacher_forward(images, decoder_ids, decoder_mask):
    patches = TEACHER_ENCODER(
        pixel_values=images
    ).last_hidden_state
    assert patches.shape[1:] == (196, 768)
    memory = TEACHER_ADAPTER(patches)
    return TEACHER_DECODER(
        decoder_ids, memory, decoder_mask
    )


def student_forward(images, decoder_ids, decoder_mask):
    memory = STUDENT_ENCODER(images)
    assert memory.shape[1:] == (49, 256)
    return STUDENT_DECODER(
        decoder_ids, memory, decoder_mask
    )


def set_student_train_mode():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    frozen_batchnorm = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(
            module, nn.modules.batchnorm._BatchNorm
        ):
            module.eval()
            frozen_batchnorm += 1
    assert frozen_batchnorm == 27
    return frozen_batchnorm


teacher_parameter_count = sum(
    parameter.numel()
    for module in [
        TEACHER_ENCODER,
        TEACHER_ADAPTER,
        TEACHER_DECODER,
    ]
    for parameter in module.parameters()
)
student_parameter_count = sum(
    parameter.numel()
    for module in [STUDENT_ENCODER, STUDENT_DECODER]
    for parameter in module.parameters()
)
assert student_parameter_count == 10_925_684
assert all(
    not parameter.requires_grad
    for module in [
        TEACHER_ENCODER,
        TEACHER_ADAPTER,
        TEACHER_DECODER,
    ]
    for parameter in module.parameters()
)

smoke = {
    key: (
        value[:2].to(device)
        if isinstance(value, torch.Tensor)
        else value
    )
    for key, value in FIXED_BATCH.items()
}
with torch.no_grad():
    teacher_logits = teacher_forward(
        smoke["teacher_images"],
        smoke["decoder_input_ids"],
        smoke["decoder_attention_mask"],
    )
    student_logits = student_forward(
        smoke["student_images"],
        smoke["decoder_input_ids"],
        smoke["decoder_attention_mask"],
    )
assert teacher_logits.shape == student_logits.shape
assert teacher_logits.shape[-1] == 10080
assert torch.isfinite(teacher_logits).all()
assert torch.isfinite(student_logits).all()

write_json(
    AUDIT_DIR / "phase4o_cell04_model_load.json",
    {
        "audit_version": "phase4o_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher": "SigLIP ViT-B/16 + shared decoder",
        "student": "TinyViT-5M + shared decoder",
        "teacher_checkpoint_sha256": sha256_file(
            TEACHER_C_BEST_CHECKPOINT
        ),
        "student_encoder_checkpoint_sha256": sha256_file(
            S3_INITIAL_CHECKPOINT
        ),
        "student_decoder_checkpoint_sha256": sha256_file(
            SHARED_DECODER_INITIAL_CHECKPOINT
        ),
        "teacher_parameters": teacher_parameter_count,
        "student_parameters": student_parameter_count,
        "teacher_visual_tokens": 196,
        "student_visual_tokens": 49,
        "teacher_frozen": True,
        "teacher_logits_shape": list(teacher_logits.shape),
        "student_logits_shape": list(student_logits.shape),
        "strict_loads_passed": 5,
        "network_download": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04O CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher strict load       : PASSED")
print("Teacher component loads   : 3 / 3 PASSED")
print("Student component loads   : 2 / 2 PASSED")
print("Teacher frozen            : True")
print("Teacher visual memory     : [B,196,256]")
print("Student visual memory     : [B,49,256]")
print("Teacher parameters        :", teacher_parameter_count)
print("Student parameters        :", student_parameter_count)
print("Teacher logits            :", list(teacher_logits.shape))
print("Student logits            :", list(student_logits.shape))
print("Vocabulary logits         : 10080")
print("Network download          : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : verify KD loss semantics")


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

PHASE 04O CELL 4 STATUS
Status                    : PASSED
Teacher strict load       : PASSED
Teacher component loads   : 3 / 3 PASSED
Student component loads   : 2 / 2 PASSED
Teacher frozen            : True
Teacher visual memory     : [B,196,256]
Student visual memory     : [B,49,256]
Teacher parameters        : 98891104
Student parameters        : 10925684
Teacher logits            : [2, 13, 10080]
Student logits            : [2, 13, 10080]
Vocabulary logits         : 10080
Network download          : False
Full training started     : False
Test Set status           : SEALED
Next                      : verify KD loss semantics


## Cell 5 — KD loss semantic unit tests

Cell này chứng minh:

- mask PAD đúng;
- `alpha=0` tái tạo CE-only;
- logits giống nhau cho KL xấp xỉ 0;
- hệ số `T²` được áp dụng;
- Teacher logits luôn được detach.


In [5]:
CELL_STARTED_AT = time.perf_counter()
TEMPERATURE = float(LOSS_CONTRACT["loss"]["temperature"])
ALPHA = float(LOSS_CONTRACT["loss"]["alpha"])
PAD_ID = int(LOSS_CONTRACT["compatibility"]["pad_id"])
assert TEMPERATURE == 4.0
assert ALPHA == 0.5
assert PAD_ID == 0


def kd_loss_components(
    student_logits,
    teacher_logits,
    labels,
    alpha=ALPHA,
    temperature=TEMPERATURE,
):
    valid = labels.ne(PAD_ID)
    valid_count = valid.sum()
    assert int(valid_count) > 0
    ce_tokens = F.cross_entropy(
        student_logits.float().reshape(-1, 10080),
        labels.reshape(-1),
        ignore_index=PAD_ID,
        reduction="sum",
    )
    ce = ce_tokens / valid_count
    student_log_probs = F.log_softmax(
        student_logits.float() / temperature, dim=-1
    )
    teacher_probs = F.softmax(
        teacher_logits.detach().float() / temperature,
        dim=-1,
    )
    kl_per_token = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="none",
    ).sum(dim=-1)
    kl = (
        (kl_per_token * valid).sum()
        / valid_count
        * (temperature ** 2)
    )
    total = (1.0 - alpha) * ce + alpha * kl
    return {
        "total": total,
        "ce": ce,
        "kd_kl_t2": kl,
        "valid_tokens": int(valid_count),
    }


generator = torch.Generator(device=device).manual_seed(42004)
synthetic = torch.randn(
    2, 5, 10080, generator=generator, device=device
)
synthetic_labels = torch.tensor(
    [
        [4, 5, 6, 2, 0],
        [7, 8, 9, 10, 2],
    ],
    device=device,
)
identical = kd_loss_components(
    synthetic, synthetic.clone(), synthetic_labels
)
assert abs(float(identical["kd_kl_t2"])) < 1e-5

alpha_zero = kd_loss_components(
    synthetic,
    synthetic * 0.5,
    synthetic_labels,
    alpha=0.0,
)
assert torch.equal(alpha_zero["total"], alpha_zero["ce"])

changed_student = synthetic.clone()
changed_teacher = (synthetic * 0.5).clone()
changed_student[0, -1, :] = 100.0
changed_teacher[0, -1, :] = -100.0
padding_a = kd_loss_components(
    synthetic, synthetic * 0.5, synthetic_labels
)
padding_b = kd_loss_components(
    changed_student, changed_teacher, synthetic_labels
)
assert torch.equal(padding_a["total"], padding_b["total"])

LOSS_IMPLEMENTATION_CONTRACT = (
    CONFIG_DIR / "phase4o_verified_kd_loss_implementation.json"
)
write_json(
    LOSS_IMPLEMENTATION_CONTRACT,
    {
        "contract_version": "phase4o_kd_loss_verified_v1",
        "created_at_utc": now_utc(),
        "status": "verified",
        "formula": (
            "(1-alpha)*CE + alpha*T^2*KL("
            "teacher_probs || student_probs)"
        ),
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "pad_id": PAD_ID,
        "identical_logits_kl_absolute": abs(
            float(identical["kd_kl_t2"])
        ),
        "alpha_zero_exact_ce": True,
        "teacher_logits_detached": True,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    AUDIT_DIR / "phase4o_cell05_loss_semantics.json",
    {
        "audit_version": "phase4o_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "identical_logits_near_zero_kl": True,
        "alpha_zero_reproduces_ce": True,
        "padding_mask_verified": True,
        "temperature_squared_scale": True,
        "teacher_logits_detached": True,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04O CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Temperature / alpha       :", TEMPERATURE, "/", ALPHA)
print("PAD masking               : PASSED")
print("Identical-logit KL        :", float(identical["kd_kl_t2"]))
print("Alpha=0 equals CE         : EXACT")
print("Teacher logits detached   : True")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : real-batch forward/backward audit")


PHASE 04O CELL 5 STATUS
Status                    : PASSED
Temperature / alpha       : 4.0 / 0.5
PAD masking               : PASSED
Identical-logit KL        : 7.232754342112457e-07
Alpha=0 equals CE         : EXACT
Teacher logits detached   : True
Full training started     : False
Test Set status           : SEALED
Next                      : real-batch forward/backward audit


## Cell 6 — Real-batch forward, loss và backward

Teacher chạy `no_grad`; Student nhận gradient. Cell kiểm tra CE, KL và tổng
loss hữu hạn, gradient Student hữu hạn/khác 0, Teacher hoàn toàn không có
gradient và eval forward lặp lại chính xác.


In [6]:
CELL_STARTED_AT = time.perf_counter()
batch = {
    key: (
        value.to(device)
        if isinstance(value, torch.Tensor)
        else value
    )
    for key, value in FIXED_BATCH.items()
}

for module in [
    TEACHER_ENCODER,
    TEACHER_ADAPTER,
    TEACHER_DECODER,
]:
    module.eval()
set_student_train_mode()
for module in [STUDENT_ENCODER, STUDENT_DECODER]:
    module.zero_grad(set_to_none=True)

with torch.no_grad():
    fixed_teacher_logits = teacher_forward(
        batch["teacher_images"],
        batch["decoder_input_ids"],
        batch["decoder_attention_mask"],
    ).detach()

student_logits = student_forward(
    batch["student_images"],
    batch["decoder_input_ids"],
    batch["decoder_attention_mask"],
)
components = kd_loss_components(
    student_logits,
    fixed_teacher_logits,
    batch["labels"],
)
assert all(
    torch.isfinite(components[name])
    for name in ["total", "ce", "kd_kl_t2"]
)
components["total"].backward()

student_gradients = [
    parameter.grad
    for module in [STUDENT_ENCODER, STUDENT_DECODER]
    for parameter in module.parameters()
    if parameter.requires_grad
]
assert student_gradients
assert all(gradient is not None for gradient in student_gradients)
assert all(
    torch.isfinite(gradient).all()
    for gradient in student_gradients
)
student_gradient_norm = math.sqrt(
    sum(
        float(gradient.detach().float().pow(2).sum())
        for gradient in student_gradients
    )
)
assert student_gradient_norm > 0.0
assert all(
    parameter.grad is None
    for module in [
        TEACHER_ENCODER,
        TEACHER_ADAPTER,
        TEACHER_DECODER,
    ]
    for parameter in module.parameters()
)

STUDENT_ENCODER.eval()
STUDENT_DECODER.eval()
with torch.no_grad():
    repeat_a = student_forward(
        batch["student_images"][:2],
        batch["decoder_input_ids"][:2],
        batch["decoder_attention_mask"][:2],
    )
    repeat_b = student_forward(
        batch["student_images"][:2],
        batch["decoder_input_ids"][:2],
        batch["decoder_attention_mask"][:2],
    )
repeat_error = float((repeat_a - repeat_b).abs().max())
assert repeat_error == 0.0

for module in [STUDENT_ENCODER, STUDENT_DECODER]:
    module.zero_grad(set_to_none=True)

write_json(
    AUDIT_DIR / "phase4o_cell06_forward_backward.json",
    {
        "audit_version": "phase4o_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "batch_images": 8,
        "valid_tokens": components["valid_tokens"],
        "ce": float(components["ce"].detach()),
        "kd_kl_t2": float(components["kd_kl_t2"].detach()),
        "combined_loss": float(components["total"].detach()),
        "student_gradient_l2": student_gradient_norm,
        "student_gradients_finite": True,
        "teacher_gradients_absent": True,
        "eval_repeat_error": repeat_error,
        "optimizer_step_performed": False,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04O CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("CE loss                   :", float(components["ce"].detach()))
print("KD KL*T^2                 :", float(components["kd_kl_t2"].detach()))
print("Combined loss             :", float(components["total"].detach()))
print("Valid tokens              :", components["valid_tokens"])
print("Student gradient L2       :", student_gradient_norm)
print("Teacher gradients         : NONE")
print("Eval repeat error         :", repeat_error)
print("Optimizer step performed  : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : one-batch KD overfit gate")


PHASE 04O CELL 6 STATUS
Status                    : PASSED
CE loss                   : 11.076396942138672
KD KL*T^2                 : 4.567009449005127
Combined loss             : 7.82170295715332
Valid tokens              : 84
Student gradient L2       : 6.637286689318635
Teacher gradients         : NONE
Eval repeat error         : 0.0
Optimizer step performed  : False
Full training started     : False
Test Set status           : SEALED
Next                      : one-batch KD overfit gate


## Cell 7 — One-batch KD overfit gate

Chạy tối đa 150 sanity-check steps trên đúng tám ảnh cố định. Gate yêu cầu:

- combined loss ratio ≤ 0.35;
- token accuracy ≥ 0.75;
- CE và KD component đều giảm;
- mọi gradient hữu hạn;
- sau audit phải khôi phục chính xác Student initialization.

Overfit weights không được lưu.


In [7]:
CELL_STARTED_AT = time.perf_counter()


def restore_student_initial():
    STUDENT_ENCODER.load_state_dict(
        STUDENT_ENCODER_INITIAL_STATE, strict=True
    )
    STUDENT_DECODER.load_state_dict(
        STUDENT_DECODER_INITIAL_STATE, strict=True
    )
    for module in [STUDENT_ENCODER, STUDENT_DECODER]:
        module.zero_grad(set_to_none=True)
    STUDENT_ENCODER.train()
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()


def token_accuracy(logits, labels):
    valid = labels.ne(PAD_ID)
    predicted = logits.argmax(dim=-1)
    return float(
        (predicted.eq(labels) & valid).sum() / valid.sum()
    )


def evaluate_fixed_student():
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    with torch.no_grad():
        logits = student_forward(
            batch["student_images"],
            batch["decoder_input_ids"],
            batch["decoder_attention_mask"],
        )
        losses = kd_loss_components(
            logits, fixed_teacher_logits, batch["labels"]
        )
    return {
        "total": float(losses["total"].detach()),
        "ce": float(losses["ce"].detach()),
        "kd": float(losses["kd_kl_t2"].detach()),
        "accuracy": token_accuracy(logits, batch["labels"]),
    }


restore_student_initial()
initial_eval = evaluate_fixed_student()
backbone_parameters = list(
    STUDENT_ENCODER.backbone.parameters()
)
backbone_ids = {id(parameter) for parameter in backbone_parameters}
adapter_parameters = [
    parameter
    for parameter in STUDENT_ENCODER.parameters()
    if id(parameter) not in backbone_ids
]
decoder_parameters = list(STUDENT_DECODER.parameters())
optimizer = torch.optim.AdamW(
    [
        {"params": backbone_parameters, "lr": 2e-4},
        {"params": adapter_parameters, "lr": 1e-3},
        {"params": decoder_parameters, "lr": 1e-3},
    ],
    weight_decay=0.0,
)

curve = []
maximum_steps = 150
minimum_steps = 50
stopped_early = False
for step in range(1, maximum_steps + 1):
    set_student_train_mode()
    optimizer.zero_grad(set_to_none=True)
    logits = student_forward(
        batch["student_images"],
        batch["decoder_input_ids"],
        batch["decoder_attention_mask"],
    )
    losses = kd_loss_components(
        logits, fixed_teacher_logits, batch["labels"]
    )
    assert torch.isfinite(losses["total"])
    losses["total"].backward()
    gradients = [
        parameter.grad
        for group in optimizer.param_groups
        for parameter in group["params"]
    ]
    assert all(gradient is not None for gradient in gradients)
    assert all(
        torch.isfinite(gradient).all()
        for gradient in gradients
    )
    gradient_norm = float(
        torch.nn.utils.clip_grad_norm_(
            list(STUDENT_ENCODER.parameters())
            + list(STUDENT_DECODER.parameters()),
            max_norm=1.0,
        )
    )
    optimizer.step()
    row = {
        "step": step,
        "combined_loss": float(losses["total"].detach()),
        "ce": float(losses["ce"].detach()),
        "kd_kl_t2": float(losses["kd_kl_t2"].detach()),
        "token_accuracy": token_accuracy(
            logits, batch["labels"]
        ),
        "gradient_norm": gradient_norm,
    }
    curve.append(row)
    if step == 1 or step % 25 == 0:
        print(
            f"step={step:03d} | loss={row['combined_loss']:.6f} | "
            f"ce={row['ce']:.6f} | kd={row['kd_kl_t2']:.6f} | "
            f"acc={row['token_accuracy']:.4f}"
        )
    if step >= minimum_steps and step % 10 == 0:
        current = evaluate_fixed_student()
        ratio = current["total"] / initial_eval["total"]
        if ratio <= 0.35 and current["accuracy"] >= 0.75:
            stopped_early = True
            break

final_eval = evaluate_fixed_student()
final_ratio = final_eval["total"] / initial_eval["total"]
assert final_ratio <= 0.35
assert final_eval["accuracy"] >= 0.75
assert final_eval["ce"] < initial_eval["ce"]
assert final_eval["kd"] < initial_eval["kd"]

curve_path = RESULT_DIR / "phase4o_one_batch_overfit_curve.csv"
with curve_path.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle, fieldnames=list(curve[0])
    )
    writer.writeheader()
    writer.writerows(curve)

del optimizer
restore_student_initial()
restored_eval = evaluate_fixed_student()
for key in ["total", "ce", "kd", "accuracy"]:
    assert restored_eval[key] == initial_eval[key], (
        key, restored_eval[key], initial_eval[key]
    )

write_json(
    AUDIT_DIR / "phase4o_cell07_one_batch_overfit.json",
    {
        "audit_version": "phase4o_cell07_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "images": 8,
        "augmentation": "disabled",
        "completed_steps": len(curve),
        "maximum_steps": maximum_steps,
        "stopped_early": stopped_early,
        "initial": initial_eval,
        "final": final_eval,
        "final_loss_ratio": final_ratio,
        "token_accuracy_gate": 0.75,
        "loss_ratio_gate": 0.35,
        "teacher_frozen": True,
        "student_initial_restore_exact": True,
        "overfit_weights_saved": False,
        "optimizer_steps": "sanity_check_only",
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04O CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Completed steps           :", len(curve))
print("Stopped early             :", stopped_early)
print("Initial combined loss     :", initial_eval["total"])
print("Final combined loss       :", final_eval["total"])
print("Final loss ratio          :", final_ratio)
print("Final token accuracy      :", final_eval["accuracy"])
print("CE reduced                : True")
print("KD component reduced      : True")
print("Initial restore exact     : True")
print("Overfit weights saved     : False")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : exact-resume equivalence")


step=001 | loss=7.760705 | ce=10.958049 | kd=4.563362 | acc=0.0000
step=025 | loss=0.827026 | ce=0.078607 | kd=1.575445 | acc=1.0000
step=050 | loss=0.382311 | ce=0.037759 | kd=0.726863 | acc=1.0000
PHASE 04O CELL 7 STATUS
Status                    : PASSED
Completed steps           : 50
Stopped early             : True
Initial combined loss     : 7.916995048522949
Final combined loss       : 0.3498916029930115
Final loss ratio          : 0.044195000861885055
Final token accuracy      : 1.0
CE reduced                : True
KD component reduced      : True
Initial restore exact     : True
Overfit weights saved     : False
Full training started     : False
Test Set status           : SEALED
Next                      : exact-resume equivalence


## Cell 8 — FP16 exact-resume equivalence

So sánh bitwise:

- nhánh liên tục: hai optimizer steps;
- nhánh resume: một step, atomic save/load, rồi một step.

Kiểm tra model, optimizer, scheduler, GradScaler, RNG và loss sequence. File
checkpoint tạm phải được xóa, sau đó Student trở lại initialization.


In [8]:
CELL_STARTED_AT = time.perf_counter()


def clone_cpu(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: clone_cpu(item) for key, item in value.items()}
    if isinstance(value, list):
        return [clone_cpu(item) for item in value]
    if isinstance(value, tuple):
        return tuple(clone_cpu(item) for item in value)
    return copy.deepcopy(value)


def nested_exact(left, right):
    if isinstance(left, torch.Tensor):
        return (
            isinstance(right, torch.Tensor)
            and left.dtype == right.dtype
            and left.shape == right.shape
            and torch.equal(left.cpu(), right.cpu())
        )
    if isinstance(left, dict):
        return (
            isinstance(right, dict)
            and left.keys() == right.keys()
            and all(
                nested_exact(left[key], right[key])
                for key in left
            )
        )
    if isinstance(left, (list, tuple)):
        return (
            isinstance(right, type(left))
            and len(left) == len(right)
            and all(
                nested_exact(a, b) for a, b in zip(left, right)
            )
        )
    return left == right


def build_resume_optimizer():
    backbone = list(STUDENT_ENCODER.backbone.parameters())
    backbone_ids_local = {id(parameter) for parameter in backbone}
    adapter = [
        parameter
        for parameter in STUDENT_ENCODER.parameters()
        if id(parameter) not in backbone_ids_local
    ]
    decoder = list(STUDENT_DECODER.parameters())
    optimizer_local = torch.optim.AdamW(
        [
            {"params": backbone, "lr": 1e-4},
            {"params": adapter, "lr": 3e-4},
            {"params": decoder, "lr": 3e-4},
        ],
        weight_decay=1e-4,
    )
    scheduler_local = torch.optim.lr_scheduler.LambdaLR(
        optimizer_local,
        lr_lambda=lambda step: min((step + 1) / 10.0, 1.0),
    )
    scaler_local = torch.amp.GradScaler(
        "cuda", enabled=True, init_scale=1024.0
    )
    return optimizer_local, scheduler_local, scaler_local


resume_batch = {
    key: (
        value[:4]
        if isinstance(value, torch.Tensor)
        else value
    )
    for key, value in batch.items()
}
resume_teacher_logits = fixed_teacher_logits[:4]


def amp_optimizer_step(optimizer_local, scheduler_local, scaler_local):
    set_student_train_mode()
    optimizer_local.zero_grad(set_to_none=True)
    with torch.amp.autocast(
        device_type="cuda", dtype=torch.float16
    ):
        logits = student_forward(
            resume_batch["student_images"],
            resume_batch["decoder_input_ids"],
            resume_batch["decoder_attention_mask"],
        )
        losses = kd_loss_components(
            logits,
            resume_teacher_logits,
            resume_batch["labels"],
        )
    assert torch.isfinite(losses["total"])
    scaler_local.scale(losses["total"]).backward()
    scaler_local.unscale_(optimizer_local)
    gradient_norm = torch.nn.utils.clip_grad_norm_(
        list(STUDENT_ENCODER.parameters())
        + list(STUDENT_DECODER.parameters()),
        max_norm=1.0,
    )
    assert torch.isfinite(gradient_norm)
    old_scale = scaler_local.get_scale()
    scaler_local.step(optimizer_local)
    scaler_local.update()
    new_scale = scaler_local.get_scale()
    assert new_scale >= old_scale
    scheduler_local.step()
    return {
        "loss": float(losses["total"].detach()),
        "ce": float(losses["ce"].detach()),
        "kd": float(losses["kd_kl_t2"].detach()),
        "gradient_norm": float(gradient_norm),
        "scale": float(new_scale),
    }


restore_student_initial()
set_deterministic(SEED + 404)
continuous_optimizer, continuous_scheduler, continuous_scaler = (
    build_resume_optimizer()
)
continuous_results = [
    amp_optimizer_step(
        continuous_optimizer,
        continuous_scheduler,
        continuous_scaler,
    )
    for _ in range(2)
]
continuous_encoder = clone_cpu(STUDENT_ENCODER.state_dict())
continuous_decoder = clone_cpu(STUDENT_DECODER.state_dict())
continuous_optimizer_state = clone_cpu(
    continuous_optimizer.state_dict()
)
continuous_scheduler_state = clone_cpu(
    continuous_scheduler.state_dict()
)
continuous_scaler_state = clone_cpu(
    continuous_scaler.state_dict()
)
continuous_cpu_rng = torch.get_rng_state().clone()
continuous_cuda_rng = [
    state.clone() for state in torch.cuda.get_rng_state_all()
]

restore_student_initial()
set_deterministic(SEED + 404)
resume_optimizer, resume_scheduler, resume_scaler = (
    build_resume_optimizer()
)
resume_results = [
    amp_optimizer_step(
        resume_optimizer, resume_scheduler, resume_scaler
    )
]
TEMPORARY_CHECKPOINT = (
    CHECKPOINT_DIR / "temporary_exact_resume.pt"
)
temporary_write = Path(str(TEMPORARY_CHECKPOINT) + ".tmp")
torch.save(
    {
        "encoder_state": clone_cpu(STUDENT_ENCODER.state_dict()),
        "decoder_state": clone_cpu(STUDENT_DECODER.state_dict()),
        "optimizer_state": resume_optimizer.state_dict(),
        "scheduler_state": resume_scheduler.state_dict(),
        "scaler_state": resume_scaler.state_dict(),
        "cpu_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
        "completed_steps": 1,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
    temporary_write,
)
os.replace(temporary_write, TEMPORARY_CHECKPOINT)

restore_student_initial()
loaded_optimizer, loaded_scheduler, loaded_scaler = (
    build_resume_optimizer()
)
with torch.serialization.safe_globals([TorchVersion]):
    resume_checkpoint = torch.load(
        TEMPORARY_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
assert resume_checkpoint["test_set_status"] == "sealed"
assert resume_checkpoint["test_set_used"] is False
STUDENT_ENCODER.load_state_dict(
    resume_checkpoint["encoder_state"], strict=True
)
STUDENT_DECODER.load_state_dict(
    resume_checkpoint["decoder_state"], strict=True
)
loaded_optimizer.load_state_dict(
    resume_checkpoint["optimizer_state"]
)
loaded_scheduler.load_state_dict(
    resume_checkpoint["scheduler_state"]
)
loaded_scaler.load_state_dict(
    resume_checkpoint["scaler_state"]
)
torch.set_rng_state(resume_checkpoint["cpu_rng"])
torch.cuda.set_rng_state_all(resume_checkpoint["cuda_rng"])
resume_results.append(
    amp_optimizer_step(
        loaded_optimizer, loaded_scheduler, loaded_scaler
    )
)

assert continuous_results == resume_results
assert nested_exact(
    continuous_encoder, STUDENT_ENCODER.state_dict()
)
assert nested_exact(
    continuous_decoder, STUDENT_DECODER.state_dict()
)
assert nested_exact(
    continuous_optimizer_state, loaded_optimizer.state_dict()
)
assert nested_exact(
    continuous_scheduler_state, loaded_scheduler.state_dict()
)
assert nested_exact(
    continuous_scaler_state, loaded_scaler.state_dict()
)
assert torch.equal(continuous_cpu_rng, torch.get_rng_state())
assert all(
    torch.equal(left, right)
    for left, right in zip(
        continuous_cuda_rng, torch.cuda.get_rng_state_all()
    )
)

TEMPORARY_CHECKPOINT.unlink()
assert not TEMPORARY_CHECKPOINT.exists()
restore_student_initial()

write_json(
    AUDIT_DIR / "phase4o_cell08_exact_resume.json",
    {
        "audit_version": "phase4o_cell08_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "continuous_steps": 2,
        "resume_steps": "1 + save/load + 1",
        "mixed_precision": "FP16 plus GradScaler",
        "loss_gradient_lr_exact": True,
        "encoder_state_exact": True,
        "decoder_state_exact": True,
        "optimizer_state_exact": True,
        "scheduler_state_exact": True,
        "gradscaler_state_exact": True,
        "rng_state_exact": True,
        "temporary_checkpoint_deleted": True,
        "student_initial_restore_exact": True,
        "optimizer_steps": "sanity_check_only",
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 04O CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Continuous / resume steps : 2 / 1+1")
print("Loss sequence exact       : True")
print("Encoder state exact       : True")
print("Decoder state exact       : True")
print("Optimizer state exact     : True")
print("Scheduler state exact     : True")
print("GradScaler state exact    : True")
print("RNG state exact           : True")
print("Temporary checkpoint      : DELETED")
print("Initial restore exact     : True")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Next                      : package Phase 04O")


PHASE 04O CELL 8 STATUS
Status                    : PASSED
Continuous / resume steps : 2 / 1+1
Loss sequence exact       : True
Encoder state exact       : True
Decoder state exact       : True
Optimizer state exact     : True
Scheduler state exact     : True
GradScaler state exact    : True
RNG state exact           : True
Temporary checkpoint      : DELETED
Initial restore exact     : True
Full training started     : False
Test Set status           : SEALED
Next                      : package Phase 04O


## Cell 9 — Package Phase 04O và authorize Phase 04P

Tạo manifest/checksum, completion marker và authorization cho Phase 04P
production-training readiness. Không giữ overfit weights hay checkpoint tạm.


In [9]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase4o_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase4o_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase4o_SHA256SUMS.txt"
COMPLETE_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_C_S3_LOGIT_KD_PREFLIGHT_COMPLETE.json"
)
AUTHORIZED_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_C_S3_LOGIT_KD_TRAINING_READINESS_AUTHORIZED.json"
)

write_json(
    AUTHORIZED_MARKER,
    {
        "marker_version": (
            "phase4_teacher_c_s3_logit_kd_"
            "training_readiness_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "experiment_id": "P4C_C_S3_LOGIT",
        "teacher_id": "C",
        "student_id": "S3",
        "kd_channel": "direct_logit",
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "verified_loss_contract": {
            "relative_path": (
                LOSS_IMPLEMENTATION_CONTRACT.relative_to(
                    OUTPUT_ROOT
                ).as_posix()
            ),
            "sha256": sha256_file(
                LOSS_IMPLEMENTATION_CONTRACT
            ),
        },
        "required_next_gates": [
            "production image-balanced dataloader",
            "AdamW warmup-cosine and FP16 GradScaler",
            "atomic last/best checkpoint",
            "teacher-forced Validation",
            "Beam-3 Validation generation",
            "discarded production pilot",
        ],
        "next_phase": (
            "Phase 04P Teacher C to S3 logit KD "
            "production training readiness"
        ),
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

artifact_paths = sorted(
    [
        path
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV,
            MANIFEST_JSON,
            SHA256SUMS,
            COMPLETE_MARKER,
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase4o_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = resolve_inside(OUTPUT_ROOT, row["relative_path"])
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

write_json(
    COMPLETE_MARKER,
    {
        "marker_version": (
            "phase4_teacher_c_s3_logit_kd_preflight_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "experiment_id": "P4C_C_S3_LOGIT",
        "teacher_id": "C",
        "student_id": "S3",
        "kd_channel": "direct_logit",
        "temperature": TEMPERATURE,
        "alpha": ALPHA,
        "checkpoint_strict_loads_passed": 5,
        "loss_semantics_passed": True,
        "forward_backward_passed": True,
        "one_batch_overfit_passed": True,
        "exact_resume_passed": True,
        "phase04p_training_readiness_authorized": True,
        "authorization_marker": {
            "relative_path": AUTHORIZED_MARKER.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(AUTHORIZED_MARKER),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
            },
            "json": {
                "relative_path": MANIFEST_JSON.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_JSON),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "manifest_rows": len(manifest_rows),
        "manifest_verification": "all_passed",
        "sanity_optimizer_steps_only": True,
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 04O CELL 9 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher / Student         : C / S3")
print("KD channel                : direct logit")
print("Temperature / alpha       :", TEMPERATURE, "/", ALPHA)
print("Strict checkpoint loads   : 5 / 5 PASSED")
print("Loss semantics            : PASSED")
print("Forward/backward          : PASSED")
print("One-batch overfit         : PASSED")
print("Exact resume              : PASSED")
print("Manifest rows             :", len(manifest_rows))
print("Manifest verification     : ALL PASSED")
print("Phase 04P readiness       : AUTHORIZED")
print("Full training started     : False")
print("Test Set status           : SEALED")
print(
    "Elapsed                   :",
    round(time.perf_counter() - CELL_STARTED_AT, 2),
    "giây",
)
print("Next                      : Save Version, then Phase 04P")


PHASE 04O CELL 9 STATUS
Status                    : COMPLETE
Teacher / Student         : C / S3
KD channel                : direct logit
Temperature / alpha       : 4.0 / 0.5
Strict checkpoint loads   : 5 / 5 PASSED
Loss semantics            : PASSED
Forward/backward          : PASSED
One-batch overfit         : PASSED
Exact resume              : PASSED
Manifest rows             : 11
Manifest verification     : ALL PASSED
Phase 04P readiness       : AUTHORIZED
Full training started     : False
Test Set status           : SEALED
Elapsed                   : 0.01 giây
Next                      : Save Version, then Phase 04P
